<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.2-power-grid-stability-prediction/Ex12.2_03_graph_network_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.2 · Notebook 03 — The Graph Network

**Paired with L12.2 · Prediction of Power Grid Stability**

Notebook 02 told the model which line was out by setting a bit. This notebook
shows it the network instead.

That is the whole change. The node features are the same six channels per bus.
The target is the same critical clearing time. The only difference is that the
adjacency, which the dense model never saw, is now an **input** — and it changes
from case to case, because the contingency *is* the change.

Three results come out of it.

1. **A line out is a different input, not a different label.** With line 0
   removed, buses 0 and 1 are no longer neighbours; that is expressed in the
   same variables as every other neighbourhood, so the model can generalise
   across it.
2. **Depth is a physical quantity.** The network has diameter 3, and 4 when the
   tie is out. Fewer layers than that and a machine is structurally incapable
   of feeling part of the grid.
3. **Permuting the bus numbering does not move the prediction.** Not
   approximately, not after training — exactly, by construction, and you will
   measure it.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.2-power-grid-stability-prediction/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
import os
os.makedirs(pb.RESULTS, exist_ok=True)

data = pb.build_dataset(n_ops=180, seed=12)
cases = pb.contingencies()

X, A, onehot = data["X"], data["A"], data["onehot"]
cct, op_id, cont_id = data["cct"], data["op_id"], data["cont_id"]

n_train_ops = 144
train = op_id < n_train_ops
test = ~train

nb02 = pb.load("nb02_dense")
pred_dense_test = nb02["pred_test"]
lin_test = nb02["lin_test"]
print(f"  notebook 02 dense  test MAE {np.abs(cct[test]-pred_dense_test).mean()*1e3:6.2f} ms")
print(f"  notebook 02 linear test MAE {np.abs(cct[test]-lin_test).mean()*1e3:6.2f} ms")

---

## 1 · The adjacency, per case

In Ex_05 notebook 03 the adjacency was one fixed `(6, 6)` matrix, shared by
every case, and it lived outside the batch. Here it is `(n, 6, 6)` — one graph
per case — and that single change of shape is the exercise.

In [ ]:
A_hat = pb.normalised_adjacency(A)
check_shape("normalised adjacency, per case", A_hat, (1080, 6, 6))

print()
print("  distinct topologies in the set:",
      len(np.unique(A.reshape(len(A), -1), axis=0)))
print()
for k, c in enumerate(cases):
    row = int(np.argmax(cont_id == c["index"]))
    print(f"  {c['label']:<46s} degrees {A[row].sum(axis=1).astype(int)}")

print()
print("  A_hat row sums, intact  :", np.round(A_hat[0].sum(axis=1), 4))
print("  A_hat row sums, line 0 out:",
      np.round(A_hat[int(np.argmax(cont_id == 1))].sum(axis=1), 4))
print()
print("  A_hat is symmetric everywhere:",
      bool(np.allclose(A_hat, np.transpose(A_hat, (0, 2, 1)))))

**What you should see.** A PASS on `(1080, 6, 6)`, **six distinct topologies**
— the intact network for the base case plus one graph per non-islanding outage —
and degree vectors that differ from each other in exactly the two entries you
would expect.

The degree vectors show the topology change directly: `[2 2 2 3 2 1]` intact,
and `[1 1 2 3 2 1]` with line 0 out — buses 0 and 1 each lose a neighbour.
Nothing about that is a bit flag; it is a smaller number of neighbours to
average over, and every layer downstream sees it as such.

**The graph does not carry everything about a contingency, and it is not
supposed to.** Contingencies 0 and 2 both put the fault at bus 1 and differ only
in topology; contingencies 1 and 4 both put it at bus 0 and likewise. The fault
location is a **node feature** (`is_fault_bus`) because that is what it is — a
property of a bus, not of an edge — and the outage is an edge, so it lives in
the adjacency. Neither channel identifies a contingency on its own, and a model
that ignored either would be blind to half the set.

The row sums of $\hat A$ are not all 1 because the symmetric normalisation
$\tilde D^{-1/2}\tilde A\tilde D^{-1/2}$ is symmetric, not row-stochastic. That
is Kipf and Welling's choice and it is the one you built by hand in Ex_05
notebook 02.

---

## 2 · How deep, and why the graph answers

Notebook 00 measured the diameter: **3 hops intact, 4 with line 0 out**. One
message-passing layer moves information one hop. So:

* fewer than **3** layers and Gen east's rotor cannot be influenced by the load
  at bus 5 at all, in any operating point;
* fewer than **4** and it cannot be influenced by it *in the contingency that
  matters most*, which is exactly the wrong place to be blind.

Depth is therefore not a hyperparameter here. It is a statement about the
network, and the network is in front of you. Ex_05 notebook 03 made the same
argument with a depth sweep; you may repeat it here if you have time, and the
minimum should land in the same place.

---

## 3 · The layer

$$\mathbf{H}^{(l+1)} = \tanh\!\left(
\mathbf{H}^{(l)}\mathbf{W}_{\mathrm{self}} + \mathbf{b}
\;+\;\hat{\mathbf{A}}\,\mathbf{H}^{(l)}\mathbf{W}_{\mathrm{neigh}}\right)$$

the same layer as Ex_05 notebook 03: two weight matrices, so that a bus's own
state and the aggregate of its neighbours' get separate transformations rather
than being pooled and then transformed once.

Two things are different here and both follow from the target.

**$\hat A$ has a batch axis.** `A_hat @ H` with `A_hat` of shape `(n, 6, 6)` and
`H` of shape `(n, 6, f)` is a batched matrix product: case *i* is multiplied by
*its own* graph. In Ex_05 the same expression broadcast one shared graph over
every case. The code is identical; the shapes carry the difference.

**The output is one number for the whole graph.** The CCT is a property of the
system, not of a bus, so the node representations are **mean-pooled** before the
head. Pooling by mean rather than by sum keeps the output on the same scale if
the network grows — and running on a larger network without retraining is the
claim this architecture makes.

### Your turn

In [ ]:
import torch.nn as nn

# TODO 1 --- the message-passing layer and the screening network ------------------------------------------
# Three `...` to replace:
#   line 1  ->  self.self_lin(H) + A_hat @ self.neigh_lin(H)      own state plus aggregated neighbours
#   line 2  ->  H + Z if k > 0 else Z                              residual connection after the first layer
#   line 3  ->  self.head(H.mean(dim=-2))                          mean over BUSES (dim=-2), then one number per case
class GraphLayer(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.self_lin  = nn.Linear(n_in, n_out)                # own state
        self.neigh_lin = nn.Linear(n_in, n_out, bias=False)    # neighbours

    def forward(self, A_hat, H):
        return ...                                # <- self.self_lin(H) + A_hat @ self.neigh_lin(H)


class CCTGraphNet(nn.Module):
    def __init__(self, n_in=6, hidden=32, depth=3):
        super().__init__()
        self.layers = nn.ModuleList([
            GraphLayer(n_in if k == 0 else hidden, hidden)
            for k in range(depth)])
        self.head = MLP(n_in=hidden, n_out=1, n_hidden=32, n_layers=2)

    def forward(self, A_hat, H):
        for k, layer in enumerate(self.layers):
            Z = torch.tanh(layer(A_hat, H))
            H = ...                               # <- H + Z if k > 0 else Z
        return ...                                # <- self.head(H.mean(dim=-2))


set_seed(88)
gnn = CCTGraphNet(n_in=pb.N_FEATURE, hidden=32, depth=3)
# ------------------------------------------------------------------------------

In [ ]:
A_probe = to_tensor(A_hat[:4]).reshape(4, pb.N_BUS, pb.N_BUS)
H_probe = to_tensor(X[:4]).reshape(4, pb.N_BUS, pb.N_FEATURE)
with torch.no_grad():
    out = gnn(A_probe, H_probe)

print("shape check:", tuple(H_probe.shape), "->", tuple(out.shape))
print("parameters, yours     :", f"{parameter_count(gnn):,}")

set_seed(88)
reference = pb.ScreeningGNN(n_in=pb.N_FEATURE, hidden=32, depth=3,
                            head_hidden=32, head_layers=2)
print("parameters, reference :", f"{parameter_count(reference):,}")
print("parameters, dense (nb02):", f"{int(nb02['n_params']):,}")

**What you should see.** `(4, 6, 6) -> (4, 1)`, and **6,721** parameters if you
built the reference architecture: 4,576 in the three message-passing layers and
2,145 in the two-layer head.

Check the arithmetic yourself — it is the most convincing thing in this
notebook. The first layer maps 6 features to 32: $6\times32+32$ for the self
term and $6\times32$ for the neighbour term, 416 in all. The second and third
map 32 to 32: $32\times32+32+32\times32 = 2080$ each. The head is
$32\to32\to32\to1$, so $1056+1056+33 = 2145$.

**The number six does not appear anywhere in that count.** Every weight matrix
is indexed by *features*, never by buses. Add a seventh bus and this model runs
unchanged, with the same parameters. The dense model of notebook 02 has 42 input
weights per hidden unit, one per position; add a bus and it cannot run at all.

That is the difference the rest of this notebook measures.

---

## 4 · Training

Same schedule as notebook 02 — Adam then L-BFGS, full batch — with the same
target scaling, so the two MAE figures are comparable.

### Your turn

In [ ]:
# TODO 2 --- train the graph network ---------------------------------------------------------------------
# Three `...` to replace:
#   line 1  ->  (X - mu_x) / sd_x                     six per-channel statistics, shared by every bus
#   line 2  ->  mse(gnn(At, Ht) - yt)
#   line 3  ->  to_numpy(gnn(Ate, Hte)).ravel() * Y_SCALE
flat  = X[train].reshape(-1, pb.N_FEATURE)
mu_x  = flat.mean(axis=0)
sd_x  = flat.std(axis=0)
sd_x  = np.where(sd_x > 1e-12, sd_x, 1.0)
Xs    = ...                                       # <- (X - mu_x) / sd_x

Y_SCALE = 0.1
At  = to_tensor(A_hat[train]).reshape(-1, pb.N_BUS, pb.N_BUS)
Ht  = to_tensor(Xs[train]).reshape(-1, pb.N_BUS, pb.N_FEATURE)
yt  = to_tensor(cct[train] / Y_SCALE)
Ate = to_tensor(A_hat[test]).reshape(-1, pb.N_BUS, pb.N_BUS)
Hte = to_tensor(Xs[test]).reshape(-1, pb.N_BUS, pb.N_FEATURE)
yte = to_tensor(cct[test] / Y_SCALE)

def loss_gnn():
    return ...                                    # <- mse(gnn(At, Ht) - yt)

history_gnn = train_two_stage(gnn, loss_gnn, adam_steps=3000,
                              lbfgs_steps=120, lr=3e-3, report_every=500)

with torch.no_grad():
    pred_gnn_train = to_numpy(gnn(At, Ht)).ravel() * Y_SCALE
    pred_gnn_test  = ...                          # <- to_numpy(gnn(Ate, Hte)).ravel() * Y_SCALE
# ------------------------------------------------------------------------------

In [ ]:
plot_curves(history_gnn, title="graph network on the six-bus contingency set")
plt.show()

rows = [
    ["predict the mean", f"{np.abs(cct[test]-cct[train].mean()).mean()*1e3:.2f}",
     "—", "—"],
    ["linear, 42 inputs", f"{np.abs(cct[test]-lin_test).mean()*1e3:.2f}",
     f"{np.abs(cct[test]-lin_test).max()*1e3:.1f}", "—"],
    ["dense MLP (nb 02)", f"{np.abs(cct[test]-pred_dense_test).mean()*1e3:.2f}",
     f"{np.abs(cct[test]-pred_dense_test).max()*1e3:.1f}",
     f"{int(nb02['n_params']):,}"],
    ["graph network", f"{np.abs(cct[test]-pred_gnn_test).mean()*1e3:.2f}",
     f"{np.abs(cct[test]-pred_gnn_test).max()*1e3:.1f}",
     f"{parameter_count(gnn):,}"],
]
print(error_table(rows, ["model", "test MAE [ms]", "worst [ms]", "parameters"]))

c_gnn = pb.confusion(cct[test], pred_gnn_test)
c_dense = pb.confusion(cct[test], pred_dense_test)
print()
print(f"  {'':<18s}{'missed':>9s}{'alarms':>9s}{'recall':>9s}{'accuracy':>10s}")
for name, c in [("dense (nb 02)", c_dense), ("graph", c_gnn)]:
    print(f"  {name:<18s}{c['missed']:>9d}{c['alarms']:>9d}"
          f"{c['recall']:>9.3f}{c['accuracy']:>10.3f}")

pb.plot_parity(cct[test], pred_gnn_test, title="graph network, held-out dispatches")
plt.tight_layout(); plt.show()

**What you should see.** A table with four rows and the reference numbers
already filled in for the first two: **108.21 ms** for predicting the mean and
**20.72 ms** for the linear model, with a worst case of **84.0 ms**.

The two rows you filled in are your result and I am not going to tell you what
they should be. What is worth saying in advance is what a fair comparison looks
like.

**The graph model may well not win on MAE, and that would not be a failure of
the argument.** It has 6,721 parameters against the dense model's 15,297, it is
solving a harder optimisation, and on *this* set — six contingencies, all of
them seen in training — the dense model's one-hot is a perfectly good lookup
key. If your two MAEs are within a few milliseconds of each other, report that
plainly. Section 5 and section 6 are where the two architectures separate, and
neither of them is a question about accuracy.

If the graph model is *much* worse — say more than twice the dense MAE — check
depth first. Two layers cannot see the whole network and it shows up exactly
here.

---

## 5 · The permutation test

This is the measurement the whole architecture argument rests on, and it is
worth being precise about what is being claimed.

The dense model's prediction is a function of a 42-vector whose entries are
indexed by position. Relabel the buses — call bus 3 "bus 0" — and the same
physical system produces a different 42-vector, so the model produces a
different answer. Nothing is wrong with the model. The *representation* has a
convention in it.

The graph model's prediction is a function of $(X, \hat A)$ and is
**permutation invariant**: for any permutation matrix $P$,

$$f(PX,\;P\hat AP^{\top}) = f(X,\hat A)$$

exactly, in exact arithmetic, for every setting of the weights — before
training, after training, and for a model that has learned nothing. It is a
property of the computation, not of the fit, and no term in the loss rewarded
it.

Note that this is *invariance*, not the *equivariance* you measured in Ex_05.
There, the output was one vector per node and permuting the input permuted the
output. Here the output is a single number for the whole graph, and mean
pooling turns equivariance into invariance: the number does not move at all.

### Your turn

In [ ]:
# TODO 3 --- relabel the buses: which model notices? --------------------------------------------------------
# Three `...` to replace:
#   line 1  ->  np.einsum("ij,njk,lk->nil", Pm, A[test], Pm)         P A P^T for every test case at once
#   line 2  ->  np.abs(g_perm - pred_gnn_test).max()                 the graph model's gap
#   line 3  ->  np.abs(d_perm - pred_dense_test).max()               the dense model's gap
perm = [3, 0, 5, 2, 4, 1]
Pm   = pb.permutation_matrix(perm)

Xs_perm    = np.einsum("ij,njf->nif", Pm, Xs[test])
A_perm     = ...                                  # <- np.einsum("ij,njk,lk->nil", Pm, A[test], Pm)
A_hat_perm = pb.normalised_adjacency(A_perm)
assert np.allclose(A_perm, np.transpose(A_perm, (0, 2, 1))), "permuted adjacency must stay symmetric"
assert np.allclose(A_perm.sum(axis=(1, 2)), A[test].sum(axis=(1, 2))), "the edge count must not change"

with torch.no_grad():                             # the graph model sees the permuted graph
    g_perm = to_numpy(gnn(to_tensor(A_hat_perm).reshape(-1, 6, 6),
                          to_tensor(Xs_perm).reshape(-1, 6, pb.N_FEATURE))).ravel() * Y_SCALE

Z_perm  = pb.dense_inputs(np.einsum("ij,njf->nif", Pm, X[test]), onehot[test])   # the dense model only sees features
Zs_perm = (Z_perm - nb02["mu"]) / nb02["sd"]
with torch.no_grad():
    d_perm = to_numpy(dense(to_tensor(Zs_perm))).ravel() * Y_SCALE

gap_gnn   = ...                                   # <- np.abs(g_perm - pred_gnn_test).max()
gap_dense = ...                                   # <- np.abs(d_perm - pred_dense_test).max()
mae_gnn_perm   = np.abs(cct[test] - g_perm).mean()
mae_dense_perm = np.abs(cct[test] - d_perm).mean()
# ------------------------------------------------------------------------------

In [ ]:
print(f"  permutation used: {[3, 0, 5, 2, 4, 1]}")
print()
print(f"  graph model  max |f(PX, PAP') - f(X, A)| : {gap_gnn*1e3:12.3e} ms")
print(f"  dense model  max |f(PX)      - f(X)|     : {gap_dense*1e3:12.3f} ms")
print()
print(f"  graph model  MAE  {np.abs(cct[test]-pred_gnn_test).mean()*1e3:8.2f} ms"
      f"  ->  {mae_gnn_perm*1e3:10.2f} ms after relabelling")
print(f"  dense model  MAE  {np.abs(cct[test]-pred_dense_test).mean()*1e3:8.2f} ms"
      f"  ->  {mae_dense_perm*1e3:10.2f} ms after relabelling")
print()
print(f"  for scale, the whole spread of the labels is {cct.std()*1e3:.1f} ms")
print(f"  float64 epsilon is {np.finfo(np.float64).eps:.3e}")

**What you should see.**

For the **graph model**, a gap at the level of floating-point round-off. It will
not be exactly zero, and it should not be: floating-point addition is not
associative, so summing over neighbours in a different order gives an answer
that differs in the last bits. In float64 that is a relative error of order
`1e-16`, and against a prediction of order 200 ms it means a gap of order
`1e-14` ms or smaller. In float32 the same test would show `1e-7` and be far
less convincing, which is one reason `pinn_core` sets double precision.

For the **dense model**, a gap comparable to the entire dataset. The linear
baseline, which you can run in a second and which makes the same point without
any training at all, gives:

```
  max |f(PX) - f(X)|   2873.3 ms
  MAE  20.72 ms  ->  1709.9 ms after relabelling
  false alarms  5  ->  190   of 216 test cases
```

against a label spread of **132.5 ms**. The relabelled dense model is not
degraded; it is destroyed. It flags 190 of 216 contingencies, which is the same
information content as flagging all of them.

Three things follow, in increasing order of importance.

**The graph model's invariance is exact and was never trained for.** Every
operation in it is either applied to each bus separately with shared weights, or
is a sum over neighbours. Neither can see a bus number.

**The dense model did not "fail to learn" invariance. It cannot have it.** Its
first layer multiplies a 42-vector by a matrix whose entries are indexed by
position. Permuting the input permutes which weight each number meets. There are
720 relabellings of six buses and $n!$ in general; no amount of data covers
them.

**This is why the property is worth accuracy.** Whatever the MAE table said in
section 4, the dense model's number depends on the order somebody happened to
list the buses in a file. A model whose correctness depends on the row order of
a CSV is not a model you hand to an operator — and in a real EMS the bus
ordering does change, every time the network model is rebuilt.

L12.2 puts it in one sentence: *the model has no notion of bus number, only of
structure.*

---

## 6 · Save

In [ ]:
pb.save("nb03_graph",
        pred_train=pred_gnn_train, pred_test=pred_gnn_test,
        cct=cct, train=train, test=test, cont_id=cont_id, op_id=op_id,
        mu_x=mu_x, sd_x=sd_x, y_scale=np.array(Y_SCALE),
        adam=history_gnn["adam"], lbfgs=history_gnn["lbfgs"],
        n_params=np.array(parameter_count(gnn)),
        gap_gnn=np.array(gap_gnn), gap_dense=np.array(gap_dense))

---

## 7 · Before you move on

1. Report both permutation gaps with their units. Say what the graph model's
   number would have been in float32 and why that would have made the claim
   weaker.
2. Contingencies 0 and 2 put the fault at the same bus and differ only in the
   graph; contingencies 1 and 4 do the reverse. Explain what each of the two
   input channels is doing, and what the model would confuse if either were
   dropped.
3. Section 4 may have shown the dense model with the lower MAE. Write the two
   sentences you would use to argue for the graph model anyway, to a colleague
   who only reads the accuracy column.
4. `ScreeningGNN` has 6,721 parameters and the number 6 appears nowhere in the
   count. State what you would have to change to run it on a 400-bus network,
   and what you would have to *check* before believing its answers there.

Next: **notebook 04**, where a predicted clearing time becomes a decision about
what to simulate.